In [1]:
import pandas as pd

train_data = pd.read_csv("l3cube_mr_train.csv")
val_data = pd.read_csv("l3cube_mr_val.csv")
test_data = pd.read_csv("l3cube_mr_test.csv")

train_data.head(), val_data.head(), test_data.head()

(                 uid                                               text  \
 0  l3cube_mr_train_0           @Rohitkh76543 @PawarSpeaks तू कोण लवड्या   
 1  l3cube_mr_train_1  @mohitbharatiya_ लवड्या अगोदरच रडायला लागलास तू 😂   
 2  l3cube_mr_train_2  कुठं तरी आईघाल जा लवड्या, काही केला तरी तुझी क...   
 3  l3cube_mr_train_3  तेलतुंबड्या सारख्या च्या नाजायज पैदाइशीचा कुत्...   
 4  l3cube_mr_train_4  @TV9Marathi @BJP4Maharashtra नेता...???मादरचोद...   
 
    label_yn  
 0         1  
 1         1  
 2         1  
 3         1  
 4         1  ,
                uid                                               text  \
 0  l3cube_mr_val_0  @BhatkhalkarA तुझी लायकी काय तू बोलतो कोणा बदल...   
 1  l3cube_mr_val_1  हिंदू धर्माबद्दल अपशब्द उगारणारा मोकाट आन उस्म...   
 2  l3cube_mr_val_2  @abpmajhatv खांडक्या,विघानसभेत अजितदादा काय बो...   
 3  l3cube_mr_val_3  @ashish_jadhao @AUThackeray सरकार मधला बावळट स...   
 4  l3cube_mr_val_4  ब्राह्मणवाद इतका वाईट, निच, आणि कपटी आहे की स्...   
 
    l

In [3]:
from mahaNLP.preprocess import Preprocess

model2 = Preprocess()

import re

def remove_nondevnagari(text):
    return re.sub(r'[^\u0900-\u097F\s]', '', text)

'''
text5 = 'डाळी भारतीय थाळीमध्ये सामील असलेले मुख्य भोजन आहेत. US agriculture department, यु एस एग्रीकल्चर डिपार्टमेंट नुसार १०० ग्रॅम डाळ मध्ये 8 ते 9 ग्राम प्रोटीन असतात.'

remove_nondevnagari(text5)
output - 
'डाळी भारतीय थाळीमध्ये सामील असलेले मुख्य भोजन आहेत    यु एस एग्रीकल्चर डिपार्टमेंट नुसार १०० ग्रॅम डाळ मध्ये  ते  ग्राम प्रोटीन असतात'
'''


def preprocess_text(text):
    text = model2.remove_url(text)
    text = ' '.join(model2.remove_stopwords(text))
    text = remove_nondevnagari(text)
    return text

train_data['text'] = train_data['text'].apply(preprocess_text)
val_data['text'] = val_data['text'].apply(preprocess_text)
test_data['text'] = test_data['text'].apply(preprocess_text)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV


vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_data['text'])
y_train = train_data['label_yn']

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Initialize RandomForest and GridSearchCV
clf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(clf, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
best_clf = grid_search.best_estimator_

Fitting 3 folds for each of 216 candidates, totalling 648 fits


In [9]:
# Evaluation on Validation Data
X_val = vectorizer.transform(val_data['text'])
y_val = val_data['label_yn']

y_val_pred = best_clf.predict(X_val)

print("Validation Data Metrics:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))

# Evaluation on Test Data
X_test = vectorizer.transform(test_data['text'])
y_test = test_data['label_yn']

y_test_pred = best_clf.predict(X_test)

print("\nTest Data Metrics:")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

# Print the best parameters
print("\nBest Parameters from Grid Search:")
print(grid_search.best_params_)

Validation Data Metrics:
Accuracy: 0.7618666666666667
              precision    recall  f1-score   support

           0       0.75      0.78      0.77      1875
           1       0.77      0.74      0.76      1875

    accuracy                           0.76      3750
   macro avg       0.76      0.76      0.76      3750
weighted avg       0.76      0.76      0.76      3750


Test Data Metrics:
Accuracy: 0.7554666666666666
              precision    recall  f1-score   support

           0       0.75      0.77      0.76      1875
           1       0.76      0.74      0.75      1875

    accuracy                           0.76      3750
   macro avg       0.76      0.76      0.76      3750
weighted avg       0.76      0.76      0.76      3750


Best Parameters from Grid Search:
{'bootstrap': False, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 150}


In [35]:
from sklearn.svm import SVC

clf_svm = SVC(kernel='linear', probability=True, random_state=42)
clf_svm.fit(X_train, y_train)

y_val_pred_svm = clf_svm.predict(X_val)
y_test_pred_svm = clf_svm.predict(X_test)

print("Validation Data Metrics (SVM):")
print("Accuracy:", accuracy_score(y_val, y_val_pred_svm))
print(classification_report(y_val, y_val_pred_svm))

print("\nTest Data Metrics (SVM):")
print("Accuracy:", accuracy_score(y_test, y_test_pred_svm))
print(classification_report(y_test, y_test_pred_svm))

Validation Data Metrics (SVM):
Accuracy: 0.7904
              precision    recall  f1-score   support

           0       0.79      0.80      0.79      1875
           1       0.79      0.78      0.79      1875

    accuracy                           0.79      3750
   macro avg       0.79      0.79      0.79      3750
weighted avg       0.79      0.79      0.79      3750


Test Data Metrics (SVM):
Accuracy: 0.7802666666666667
              precision    recall  f1-score   support

           0       0.78      0.77      0.78      1875
           1       0.78      0.79      0.78      1875

    accuracy                           0.78      3750
   macro avg       0.78      0.78      0.78      3750
weighted avg       0.78      0.78      0.78      3750



In [36]:
from sklearn.linear_model import LogisticRegression

clf_lr = LogisticRegression(random_state=42)
clf_lr.fit(X_train, y_train)

y_val_pred_lr = clf_lr.predict(X_val)
y_test_pred_lr = clf_lr.predict(X_test)

print("Validation Data Metrics (Logistic Regression):")
print("Accuracy:", accuracy_score(y_val, y_val_pred_lr))
print(classification_report(y_val, y_val_pred_lr))

print("\nTest Data Metrics (Logistic Regression):")
print("Accuracy:", accuracy_score(y_test, y_test_pred_lr))
print(classification_report(y_test, y_test_pred_lr))

Validation Data Metrics (Logistic Regression):
Accuracy: 0.7968
              precision    recall  f1-score   support

           0       0.79      0.81      0.80      1875
           1       0.80      0.79      0.79      1875

    accuracy                           0.80      3750
   macro avg       0.80      0.80      0.80      3750
weighted avg       0.80      0.80      0.80      3750


Test Data Metrics (Logistic Regression):
Accuracy: 0.7848
              precision    recall  f1-score   support

           0       0.78      0.79      0.79      1875
           1       0.79      0.78      0.78      1875

    accuracy                           0.78      3750
   macro avg       0.78      0.78      0.78      3750
weighted avg       0.78      0.78      0.78      3750



In [37]:
from sklearn.neighbors import KNeighborsClassifier

clf_knn = KNeighborsClassifier(n_neighbors=29) 
clf_knn.fit(X_train, y_train)

y_val_pred_knn = clf_knn.predict(X_val)
y_test_pred_knn = clf_knn.predict(X_test)

print("Validation Data Metrics (KNN):")
print("Accuracy:", accuracy_score(y_val, y_val_pred_knn))
print(classification_report(y_val, y_val_pred_knn))

print("\nTest Data Metrics (KNN):")
print("Accuracy:", accuracy_score(y_test, y_test_pred_knn))
print(classification_report(y_test, y_test_pred_knn))

Validation Data Metrics (KNN):
Accuracy: 0.512
              precision    recall  f1-score   support

           0       0.87      0.03      0.05      1875
           1       0.51      1.00      0.67      1875

    accuracy                           0.51      3750
   macro avg       0.69      0.51      0.36      3750
weighted avg       0.69      0.51      0.36      3750


Test Data Metrics (KNN):
Accuracy: 0.5125333333333333
              precision    recall  f1-score   support

           0       0.89      0.03      0.06      1875
           1       0.51      1.00      0.67      1875

    accuracy                           0.51      3750
   macro avg       0.70      0.51      0.36      3750
weighted avg       0.70      0.51      0.36      3750

